In [ ]:
# Cell 1: load historical log output.
from pathlib import Path
from collections import deque
import json

REPO = Path('/workspace/stable-query-latent')
LOG = Path('/workspace/stable_query_latent_logs/pipeline.log')
SWEEP_MANIFEST = REPO / 'VICReg_review/heads/cloud_full_sweep_a100/sweep_manifest.json'
EMBED_MANIFEST = REPO / 'game_review_data/embedding_h5.h5.incloud_manifest.json'
TEXT_MANIFEST = REPO / 'game_review_data/build_new_gamedata/text_h5.h5.manifest.json'


def tail(path=LOG, lines=200):
    path = Path(path)
    print(f'log: {path}')
    if not path.exists():
        print('missing log file')
        return
    with path.open('r', encoding='utf-8', errors='replace') as f:
        for line in deque(f, maxlen=lines):
            print(line, end='')


def show_manifest(path):
    path = Path(path)
    print(f'
=== {path} ===')
    if not path.exists():
        print('missing')
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception as exc:
        print('bad json:', exc)
        return
    for key in ['status', 'updated_at', 'finished_at', 'epoch', 'step', 'error']:
        if key in data:
            print(f'{key}: {data[key]}')
    current = data.get('current')
    if current:
        print('current:', json.dumps(current, ensure_ascii=False))
    rows = data.get('rows')
    if isinstance(rows, list):
        done = sum(1 for row in rows if row.get('status') == 'done')
        print(f'combinations_done: {done}/{len(rows)}')


tail(lines=200)
show_manifest(TEXT_MANIFEST)
show_manifest(EMBED_MANIFEST)
show_manifest(SWEEP_MANIFEST)


In [ ]:
# Cell 2: realtime latest log output.
# Stop/interrupt this cell to stop watching. It does not stop the training job.
import time
from IPython.display import clear_output


def follow(path=LOG, lines=120, interval=5):
    path = Path(path)
    while True:
        clear_output(wait=True)
        print(f'{time.strftime("%Y-%m-%d %H:%M:%S")} | {path}')
        print('-' * 100)
        if not path.exists():
            print('missing log file')
        else:
            with path.open('r', encoding='utf-8', errors='replace') as f:
                for line in deque(f, maxlen=lines):
                    print(line, end='')
        time.sleep(interval)


follow(lines=120, interval=5)
